In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import warnings
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedGroupKFold


from Feature_extraction.feature_extractor import FeatureExtractor
from Embeddings.Models.embedding_generator import EmbeddingGenerator
from Model.model import IP_SAE_MODEL
from Embeddings.Semantics.descriptions import get_descriptions

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

In [2]:
DATASET_PATH = r"../Data/UiS4ADL/Processed/UiS4ADL_100hz.csv"
ADL_DICT_PATH = '../adl_dict.json'

# Parameters
FS = 100
WINDOW_SECONDS = 4.0
OVERLAP_RATIO = 0.0
METHOD = 'temporal_frequency'

EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
DESCRIPTION_TYPE = 'original_fadi'
USE_PROMPT = True

# Experimental Setup
LAMBDA = 0.001
DETERMINISTIC_RANKING = [17, 14, 11, 9, 24, 5, 7, 12, 10, 8, 6, 21, 23, 20, 22, 15, 13, 1, 19, 4, 16, 2, 18, 3]
ALL_CLASSES = sorted(DETERMINISTIC_RANKING)
K = 6

In [4]:
embedding_generator = EmbeddingGenerator(output_dir="../Data/Embeddings")
embeddings = embedding_generator.load_embeddings(
    model_name=EMBEDDING_MODEL, desc_type=DESCRIPTION_TYPE, 
    use_prompt=USE_PROMPT, dataset='fadi'
)

activity_labels = sorted(get_descriptions(DESCRIPTION_TYPE).keys())
class_to_embedding_idx = {label: idx for idx, label in enumerate(activity_labels)}

max_class_id = max(ALL_CLASSES)
all_embeddings_indexed = np.zeros((max_class_id + 1, embeddings.shape[1]))
for cls, idx in class_to_embedding_idx.items():
    if cls in ALL_CLASSES:
        all_embeddings_indexed[cls] = embeddings[idx]


feature_extractor = FeatureExtractor()
data_proc = pd.read_csv(DATASET_PATH)
sensor_cols = [col for col in data_proc.columns if col not in ['timestamp', 'adl', 'session', 'subject', 'fileID']]

X_proc, y_proc, fileIDs, subjects = feature_extractor.extract_features(
    data=data_proc, method=METHOD, sensor_columns=sensor_cols,
    window_seconds=WINDOW_SECONDS, overlap_ratio=OVERLAP_RATIO, fs=FS, strategy="retain_short"
)

X_proc = np.array(X_proc)
y_proc = np.array(y_proc)
groups_proc = np.array(fileIDs)

Loaded embeddings from: ../Data/Embeddings/all_minilm_l6_v2_original_fadi_prompt_fadi.npz
Model: all-MiniLM-L6-v2
Shape: (24, 384)

Extracting features: temporal_frequency
  Window: 4.0s
  Overlap: 0.0 (400 stride)
  Sampling rate: 100 Hz

Creating windows per fileID...
  Found 1445 files that are too short (30.94% of the available files)
  Created 20103 windows

Extracting features using method: temporal_frequency...
  Processing window 0/20103
                    1000/20103
                    2000/20103
                    3000/20103
                    4000/20103
                    5000/20103
                    6000/20103
                    7000/20103
                    8000/20103
                    9000/20103
                    10000/20103
                    11000/20103
                    12000/20103
                    13000/20103
                    14000/20103
                    15000/20103
                    16000/20103
                    17000/20103
               

In [5]:
# GZSL gamma tuning (Grouped Split)
def tune_gamma_gzsl(X_train, y_train, groups_train, embeddings_all, gamma_grid, n_iterations=10, fixed_lambda=0.001):
    """
    Simulates a GZSL task by withholding 5 seen classes as pseudo-unseen (approx 30%),
    to find the optimal calibration parameter gamma using the Harmonic Mean.
    Groups by recording ID (fileID) to prevent window-level data leakage.
    """
    seen_classes = np.unique(y_train)
    n_val_unseen = 5 
    
    gamma_scores = {round(g, 1): [] for g in gamma_grid}

    for i in range(n_iterations):
        rng = np.random.default_rng(seed=i * 42)
        val_unseen_classes = rng.choice(seen_classes, n_val_unseen, replace=False)
        val_seen_classes = np.array([c for c in seen_classes if c not in val_unseen_classes])

        # Grouped Instance-level split for pseudo-seen classes (to prevent segments from the same continuous recording from spanning train and test)
        mask_pseudo_seen = np.isin(y_train, val_seen_classes)
        X_pseudo_seen = X_train[mask_pseudo_seen]
        y_pseudo_seen = y_train[mask_pseudo_seen]
        groups_pseudo_seen = groups_train[mask_pseudo_seen] 

        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=i)

        train_idx, val_idx = next(sgkf.split(X_pseudo_seen, y_pseudo_seen, groups=groups_pseudo_seen))
        X_cv_train, y_cv_train = X_pseudo_seen[train_idx], y_pseudo_seen[train_idx]
        X_cv_test_seen, y_cv_test_seen = X_pseudo_seen[val_idx], y_pseudo_seen[val_idx]

        mask_pseudo_unseen = np.isin(y_train, val_unseen_classes)
        X_cv_test_unseen = X_train[mask_pseudo_unseen]
        y_cv_test_unseen = y_train[mask_pseudo_unseen]

        # Combine testing data for GZSL evaluation
        X_cv_test = np.vstack([X_cv_test_seen, X_cv_test_unseen])
        y_cv_test = np.concatenate([y_cv_test_seen, y_cv_test_unseen])

        # Fit the model on the pseudo-seen training set
        model = IP_SAE_MODEL(lambda_reg=fixed_lambda, scale_features=True)
        model.fit(X_cv_train, y_cv_train, embeddings_all, verbose=0)

        # Prepare Candidate set for evaluation
        candidate_classes = np.concatenate([val_seen_classes, val_unseen_classes])
        candidate_embeddings = embeddings_all[candidate_classes]

        for g in gamma_grid:
            g_rounded = round(g, 1)
            y_pred = model.predict_gzsl(
                X_test=X_cv_test, 
                all_class_embeddings=candidate_embeddings, 
                all_class_labels=candidate_classes, 
                seen_classes=val_seen_classes, 
                gamma=g_rounded, 
                n_runs=35
            )

            mask_s = np.isin(y_cv_test, val_seen_classes)
            mask_u = np.isin(y_cv_test, val_unseen_classes)

            acc_s = balanced_accuracy_score(y_cv_test[mask_s], y_pred[mask_s]) if np.sum(mask_s) > 0 else 0.0
            acc_u = balanced_accuracy_score(y_cv_test[mask_u], y_pred[mask_u]) if np.sum(mask_u) > 0 else 0.0

            # Compute Harmonic Mean (HM)
            hm = (2 * acc_s * acc_u) / (acc_s + acc_u) if (acc_s + acc_u) > 0 else 0.0
            gamma_scores[g_rounded].append(hm)


    best_gamma = max(gamma_scores, key=lambda k: np.mean(gamma_scores[k]))
    best_hm = np.mean(gamma_scores[best_gamma])
   
    print(f"\nBest Gamma found: {best_gamma} (Mean HM: {best_hm:.4f})")
    print(f"HM scores for Gamma {best_gamma}: {[round(score,4) for score in gamma_scores[best_gamma]]}")
    
    return best_gamma, gamma_scores

In [7]:
unseen_classes = DETERMINISTIC_RANKING[:K]
seen_classes = [c for c in ALL_CLASSES if c not in unseen_classes]

# Extract True Training Data (Only Seen classes)
train_mask = np.isin(y_proc, seen_classes)
X_train_main = X_proc[train_mask]
y_train_main = y_proc[train_mask]
groups_train_main = groups_proc[train_mask] # Extract grouping IDs for training subset

gamma_grid = np.arange(0.0, 1.1, 0.1)

best_g, all_cv_scores = tune_gamma_gzsl(
    X_train=X_train_main, 
    y_train=y_train_main, 
    groups_train=groups_train_main, # Pass the grouping array
    embeddings_all=all_embeddings_indexed, 
    gamma_grid=gamma_grid,
    n_iterations=10,
    fixed_lambda=LAMBDA
)
print(f"\nOptimal Gamma found: {best_g}")


Best Gamma found: 0.4 (Mean HM: 0.2186)
HM scores for Gamma 0.4: [0.1732, 0.1943, 0.1895, 0.1893, 0.2949, 0.2074, 0.2957, 0.1926, 0.2341, 0.2155]

Optimal Gamma found: 0.4
